## Step 6: Improved CNN Model

In this step, the CNN architecture is enhanced to address the limitations observed in the previous model. A deeper network with additional convolutional layers, dropout regularization, and increased training epochs is implemented to improve feature extraction and generalization.

These improvements allow the model to better capture complex spatial patterns in chest X-ray images, leading to more accurate pneumonia detection. This step demonstrates how architectural design and training strategy significantly impact deep learning performance.


In [17]:
#  Import libraries

import os
import cv2
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split

import tensorflow as tf
from tensorflow.keras import layers, models

In [18]:
# Load dataset

df = pd.read_csv("/kaggle/input/datasets/sairasagnak/balanced-data-rsna/balanced_data.csv")
print(df.shape)

(12024, 8)


In [19]:
# Fix paths

DATA_DIR = "/kaggle/input/datasets/iamtapendu/rsna-pneumonia-processed-dataset"
IMAGE_DIR = os.path.join(DATA_DIR, "Training", "Images")

df['image_path'] = df['patientId'].apply(lambda x: os.path.join(IMAGE_DIR, x + ".png"))

df = df[df['image_path'].apply(os.path.exists)].reset_index(drop=True)
print(df.shape)

(12024, 8)


In [20]:
#  Preprocessing
IMG_SIZE = 128

def preprocess_image(path):
    img = cv2.imread(path, 0)
    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
    img = img / 255.0
    return img

In [21]:
# Prepare dataset

df = df.sample(3000, random_state=42)   # slightly larger

images = []
labels = []

for _, row in df.iterrows():
    img = preprocess_image(row['image_path'])
    images.append(img)
    labels.append(row['Target'])

X = np.array(images)
y = np.array(labels)

X = X.reshape(-1, IMG_SIZE, IMG_SIZE, 1)

print(X.shape, y.shape)

(3000, 128, 128, 1) (3000,)


In [22]:
# Train-test split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [23]:
#  Build improved CNN

model = models.Sequential([

    layers.Conv2D(32, (3,3), activation='relu', input_shape=(IMG_SIZE, IMG_SIZE, 1)),
    layers.MaxPooling2D(2,2),

    layers.Conv2D(64, (3,3), activation='relu'),
    layers.MaxPooling2D(2,2),

    layers.Conv2D(128, (3,3), activation='relu'),
    layers.MaxPooling2D(2,2),

    layers.Flatten(),

    layers.Dense(128, activation='relu'),
    layers.Dropout(0.5),

    layers.Dense(1, activation='sigmoid')
])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
2026-05-01 10:27:33.343567: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


In [24]:
# Compile

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [25]:
#  Train

history = model.fit(
    X_train, y_train,
    epochs=10,
    validation_data=(X_test, y_test)
)

Epoch 1/10
75/75 ━━━━━━━━━━━━━━━━━━━━ 38s 487ms/step - accuracy: 0.6089 - loss: 0.6665 - val_accuracy: 0.7333 - val_loss: 0.5738
Epoch 2/10
75/75 ━━━━━━━━━━━━━━━━━━━━ 36s 486ms/step - accuracy: 0.7304 - loss: 0.5768 - val_accuracy: 0.7283 - val_loss: 0.5487
Epoch 3/10
75/75 ━━━━━━━━━━━━━━━━━━━━ 37s 488ms/step - accuracy: 0.7117 - loss: 0.5904 - val_accuracy: 0.7450 - val_loss: 0.5498
Epoch 4/10
75/75 ━━━━━━━━━━━━━━━━━━━━ 37s 488ms/step - accuracy: 0.7305 - loss: 0.5662 - val_accuracy: 0.7350 - val_loss: 0.5396
Epoch 5/10
75/75 ━━━━━━━━━━━━━━━━━━━━ 37s 487ms/step - accuracy: 0.7426 - loss: 0.5484 - val_accuracy: 0.7450 - val_loss: 0.5477
Epoch 6/10
75/75 ━━━━━━━━━━━━━━━━━━━━ 35s 473ms/step - accuracy: 0.7570 - loss: 0.5091 - val_accuracy: 0.7350 - val_loss: 0.5544
Epoch 7/10
75/75 ━━━━━━━━━━━━━━━━━━━━ 36s 474ms/step - accuracy: 0.7701 - loss: 0.4780 - val_accuracy: 0.7500 - val_loss: 0.5581
Epoch 8/10
75/75 ━━━━━━━━━━━━━━━━━━━━ 35s 466ms/step - accuracy: 0.7583 - loss: 0.4902 - val_accu

In [29]:
#  Evaluate

loss, accuracy = model.evaluate(X_test, y_test)

print("Improved CNN Accuracy:", accuracy)

19/19 ━━━━━━━━━━━━━━━━━━━━ 2s 125ms/step - accuracy: 0.7290 - loss: 0.7163
Improved CNN Accuracy: 0.7233333587646484


## Step 6: Observations

* The improved CNN model achieved an accuracy of approximately 72%, which did not show improvement over the previous CNN or baseline MLP model.
* Training accuracy increased steadily, while validation accuracy remained relatively constant, indicating overfitting.
* The model learned patterns specific to the training data but failed to generalize effectively to unseen data.
* Despite increasing model depth and training epochs, performance gains were limited due to insufficient dataset size and lack of data augmentation.
* These results demonstrate that architectural improvements alone are not sufficient; proper regularization and data diversity are critical for deep learning performance.

This step highlights the importance of generalization and motivates the use of advanced techniques such as data augmentation and transfer learning.
